In [ ]:
import re
import gc
import torch
import time
import pandas as pd

# ==========================================
# 1. UTILITAIRES TECHNIQUES
# ==========================================
def clean_memory():
    """Vide le cache mémoire GPU et force le Garbage Collector."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def generate_llm_response(model, tokenizer, system_prompt, user_prompt, max_tokens=450):
    """Génère la réponse en séparant bien le System et le User et mesure la latence."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Préparation des inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Mesure du temps d'inférence
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            # Le tokenizer gère l'attention_mask via apply_chat_template
            pad_token_id=tokenizer.eos_token_id 
        )
    
    end_time = time.time()
    latency = end_time - start_time

    # Décodage (on ne garde que ce qui a été généré après le prompt)
    raw_output = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    del inputs, outputs
    clean_memory()

    return raw_output, latency

# ==========================================
# 2. PARSEURS ET MÉTRIQUES
# ==========================================
def extract_ids(raw_output):
    """Extrait uniquement la liste des IDs depuis la balise <chunks_id>."""
    chunks_match = re.search(r"<chunks_id>(.*?)</chunks_id>", raw_output, re.DOTALL)
    if chunks_match:
        # Trouve tous les nombres dans le bloc de la balise
        return [int(n) for n in re.findall(r'\d+', chunks_match.group(1))]
    return []

def extract_answer(raw_output):
    """Extrait uniquement le texte depuis la balise <answer>."""
    answer_match = re.search(r"<answer>(.*?)</answer>", raw_output, re.DOTALL)
    return answer_match.group(1).strip() if answer_match else "Format Error (Answer missing tags)"

def calculate_metrics(gold_ids, predicted_ids):
    """Calcule Précision, Rappel, F1 et Accuracy (Exact Match)."""
    gold_set = set(gold_ids)
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0.0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0.0
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0
    accuracy = 1.0 if pred_set == gold_set else 0.0

    return {"precision": p, "recall": r, "f1": f1, "accuracy": accuracy}

# ==========================================
# 3. LE CONTRÔLEUR D'ÉVALUATION
# ==========================================
def evaluate_prompt_strategy(dataset, sys_prompt, user_prompt_template, model, tokenizer, sample_size=None, output_filename=None):
    """
    Évalue une stratégie de prompting (ex: Cite-First ou Answer-First).
    """
    results = []
    df_sample = dataset.head(sample_size) if sample_size is not None else dataset

    print(f"Démarrage de l'évaluation sur {len(df_sample)} échantillons...\n")

    for index, row in df_sample.iterrows():
        # 1. Construction du contexte structuré
        context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])
        
        formatted_user_prompt = user_prompt_template.format(
            context_text=context_text,
            question=row['question']
        )

        # 2. Inférence et Latence
        raw_output, latency = generate_llm_response(model, tokenizer, sys_prompt, formatted_user_prompt)

        # 3. Parsing des résultats
        predicted_answer = extract_answer(raw_output)
        predicted_ids = extract_ids(raw_output)

        # 4. Calcul des métriques
        metrics = calculate_metrics(row['gold_ids'], predicted_ids)

        # 5. Stockage des données
        results.append({
            "sample_id": row.get('id', index),
            "question": row['question'],
            "latency": latency,
            "gold_ids": row['gold_ids'],
            "predicted_ids": predicted_ids,
            "predicted_answer": predicted_answer,
            **metrics
        })

        # 6. Affichage en temps réel
        print(f"--- SAMPLE {index+1}/{len(df_sample)} ---")
        print(f"Latence : {latency:.2f}s")
        print(f"Verdict : {'✅ Exact Match' if metrics['accuracy'] else '❌ Différent'}")
        print(f"IDs Pred: {predicted_ids} | Gold: {row['gold_ids']}")
        print(f"F1-Score: {metrics['f1']:.2f}")
        print("-" * 40)

    # --- Synthèse Finale ---
    df_results = pd.DataFrame(results)
    avg = df_results[["precision", "recall", "f1", "accuracy", "latency"]].mean()

    print("\n" + "="*25)
    print("   SYNTHÈSE DES RÉSULTATS")
    print("="*25)
    print(f"Precision : {avg['precision']:.4f}")
    print(f"Recall    : {avg['recall']:.4f}")
    print(f"F1-Score  : {avg['f1']:.4f}")
    print(f"Accuracy  : {avg['accuracy']:.4f} (Exact Match)")
    print(f"Latence Moyenne : {avg['latency']:.2f}s")
    print("="*25 + "\n")

    if output_filename:
        df_results.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"💾 Résultats complets sauvegardés dans : {output_filename}")

    return df_results, avg.to_dict()

# ==========================================
# 4. CONFIGURATION DES PROMPTS (Rappel)
# ==========================================

# Exemple pour "Cite Then Answer"
sys_cite_then_answer = """### ROLE:
You are a precise Logic Auditor. Identify the MINIMAL logical path required to answer the question based EXCLUSIVELY on the [CONTEXT].

### CHUNK SELECTION:
1. Identify IDs ONLY if they bridge the gap between question and answer.
2. EXCLUDE general definitions or background info.
3. Think step-by-step about which IDs are strictly necessary.

### CONSTRAINTS:
- NO pronouns. Use full entity names.
- Single declarative sentence.

### OUTPUT FORMAT:
<chunks_id>[IDs only]</chunks_id>
<answer>[Sentence]</answer>"""

user_template = """### [CONTEXT]:
{context_text}

### [QUESTION]:
{question}

---
Reminder: Follow the protocol. Output the required tags."""

In [ ]:
# --- OPTION 1: CITE THEN ANSWER (Original order) ---
SYSTEM_PROMPT_CITE_THEN_ANSWER = """### ROLE:
You are a precise Logic Auditor. Your goal is to identify the MINIMAL logical path required to answer a question based EXCLUSIVELY on the provided "[CONTEXT]".

### CHUNK SELECTION PROTOCOL:
Identify an ID ONLY if it provides a specific fact required to bridge the gap between the question and the answer.
1. **Direct Evidence**: Contains the core fact requested.
2. **Logical Bridge**: Connects entities necessary to reach the conclusion.
3. **Strict Exclusion**: DO NOT cite chunks for general definitions or background info that doesn't change the final answer.
*Minimalism*: If removing an ID makes the reasoning chain collapse, keep it. Otherwise, discard it.

### INSTRUCTIONS:
- Think step-by-step about which IDs are strictly necessary before finalizing your <chunks_id> list.
- NO pronouns (he, she, they, it). Use full entity names.
- Provide a single, clear, declarative sentence.
- If context is insufficient, write "Insufficient information" in both fields.

### OUTPUT FORMAT:
<chunks_id>[Comma-separated list of numerical IDs only]</chunks_id>
<answer>[Your single, entity-rich sentence]</answer>"""

# --- OPTION 2: ANSWER THEN CITE (Inverted order) ---
SYSTEM_PROMPT_ANSWER_THEN_CITE = """### ROLE:
You are a precise QA agent with a mandatory requirement for evidentiary proof.

### INSTRUCTIONS:
1. **Response Phase**: Write a single, declarative sentence using ONLY the [CONTEXT]. 
   - NO pronouns (he, she, they, it). Use full entity names.
2. **Evidence Audit Phase**: Review your own answer and the [CONTEXT].
   - Think step-by-step about which IDs are strictly necessary to prove the facts in your sentence.
   - Ask yourself: "Can I prove this fact without this specific chunk?" If yes, do NOT include the ID.
   - EXCLUDE general definitions or redundant background information.
3. **No Hallucinations**: If context is insufficient, write "Insufficient information" in both fields.

### OUTPUT FORMAT:
<answer>[Your single, entity-rich sentence]</answer>
<chunks_id>[Comma-separated list of numerical IDs only]</chunks_id>"""

# --- USER PROMPT TEMPLATES (Common to both) ---
# Note: The Reminder is adapted to the specific flow of each prompt.

USER_PROMPT_CITE_THEN_ANSWER_TEMPLATE = """### [CONTEXT]:
{context_text}

### [QUESTION]:
{question}

---
Reminder: List the minimal <chunks_id> first, then the <answer>."""

USER_PROMPT_ANSWER_THEN_CITE_TEMPLATE = """### [CONTEXT]:
{context_text}

### [QUESTION]:
{question}

---
Reminder: Provide the <answer> first, then the minimal <chunks_id> list."""